In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [ ]:
# ── Step 1: Load Dataset ─────────────────────────────────────────────────────
# Each row = one house. MEDV is the target (median house price in $1000s).
data = pd.read_csv("HousingData.csv")
data.head()

In [ ]:
# ── Step 2: Prepare Features and Labels ──────────────────────────────────────
X = data.drop("MEDV", axis=1)   # all columns except the price
y = data["MEDV"]                 # price is what we want to predict

# Some cells have missing values — fill them with the column average
X = X.fillna(X.mean())

# 80% for training, 20% for testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
# ── Step 3: Feature Scaling ───────────────────────────────────────────────────
# Neural networks train better when all features are on the same scale (mean=0, std=1).
# fit_transform on train → only transform on test (avoid data leakage)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

In [ ]:
# ── Step 4: Build the Model ───────────────────────────────────────────────────
# Dense(64) → Dense(32): two hidden layers with ReLU (learns non-linear patterns)
# Dense(1) with NO activation: regression output — can predict any numeric value
model = Sequential()
model.add(Dense(64, activation='relu', input_shape=(X_train.shape[1],)))
model.add(Dense(32, activation='relu'))
model.add(Dense(1))   # no activation = linear output for regression

# adam: adaptive optimizer | mse: penalises large errors more | mae: easy to interpret
model.compile(optimizer='adam', loss='mean_squared_error', metrics=['mae'])


In [ ]:
# ── Step 5: Train the Model ───────────────────────────────────────────────────
# epochs=100: enough passes for the small dataset to converge
# batch_size=16: update weights after every 16 samples
# validation_split=0.2: holds out 20% of training data to monitor overfitting
history = model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=16,
    validation_split=0.2,
    verbose=1
)

# Run predictions on the unseen test set
y_pred = model.predict(X_test)

In [ ]:
# ── Step 6: Evaluate & Visualise ─────────────────────────────────────────────
mse  = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2   = r2_score(y_test, y_pred)

# R2 close to 1.0 = good. Negative = worse than predicting the mean.
print("\nModel Performance")
print("-------------------")
print(f"MSE      : {mse:.2f}")
print(f"RMSE     : {rmse:.2f}")
print(f"R2 Score : {r2:.4f}")

# Loss curve — training and validation should both go down together
plt.plot(history.history['loss'],     label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss (MSE)')
plt.title('Training vs Validation Loss')
plt.legend()
plt.show()

# Scatter: points close to the red diagonal = accurate predictions
plt.scatter(y_test, y_pred, alpha=0.7, color='steelblue')
plt.plot([y_test.min(), y_test.max()],
         [y_test.min(), y_test.max()], 'r--', label='Perfect Prediction')
plt.xlabel('Actual Price ($1000s)')
plt.ylabel('Predicted Price ($1000s)')
plt.title('Actual vs Predicted House Prices')
plt.legend()
plt.show()